# ForenSight: Colab quick start

Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom.

1. Setup (GPU check, Drive, install)
2. **Audio, no approval needed:** download ASVspoof 2019 LA, evaluate the pretrained AASIST, train the MelCNN baseline
3. **Video, after FaceForensics++ access is approved:** preprocess faces, train EfficientNet-B4

Results are written to `MyDrive/forensight/runs/`. Log each finished run in `docs/experiments.md`.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# The repo is private. Create a GitHub fine-grained token (Settings > Developer settings >
# Fine-grained tokens, repo ForenSight-FYP, Contents: read) and save it in Colab's
# Secrets panel (key icon on the left) as GITHUB_TOKEN with notebook access on.
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{TOKEN}@github.com/ForenSight-FYP/ForenSight-FYP.git"
!git clone -q $REPO_URL /content/forensight || (cd /content/forensight && git pull -q)
%cd /content/forensight
!pip install -q -e ".[dl,dev]"
!pytest -q

## 2. Audio module

### 2a. Get ASVspoof 2019 LA (~7.6 GB, open download)
The archive is cached on Drive after the first download so later sessions only unzip it.
If the link has moved, download `LA.zip` from the Edinburgh DataShare page for "ASVspoof 2019".

In [ ]:
import os
CACHE = "/content/drive/MyDrive/forensight/datasets/asvspoof2019_LA.zip"
os.makedirs(os.path.dirname(CACHE), exist_ok=True)
if not os.path.exists(CACHE):
    !wget -q --show-progress -O "$CACHE" https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip
!mkdir -p /content/asvspoof && unzip -q -n "$CACHE" -d /content/asvspoof
!ls /content/asvspoof/LA

### 2b. Pretrained AASIST on ASVspoof 2019 LA eval
This is the pipeline sanity check: the AASIST paper reports an EER of about 0.83% on this set.
If our number is close, the data loading, labels and metrics are all correct.

In [ ]:
!bash scripts/setup_aasist.sh
!python -m forensight.evaluation.eval_audio --config configs/audio_aasist.yaml "eval_sets={la19_eval: {protocol: /content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt, audio: /content/asvspoof/LA/ASVspoof2019_LA_eval/flac}}"

### 2c. Train the MelCNN baseline (about 20 epochs)

In [ ]:
# smoke run first: 1 epoch, to check everything works end to end
!python -m forensight.training.train_audio --config configs/audio_melcnn.yaml train.epochs=1 "eval_sets={}"

In [ ]:
!python -m forensight.training.train_audio --config configs/audio_melcnn.yaml "eval_sets={la19_eval: {protocol: /content/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt, audio: /content/asvspoof/LA/ASVspoof2019_LA_eval/flac}}"

The ASVspoof 2021 DF set (cross-dataset test) is added once downloaded: see `docs/datasets.md`,
then run the same commands without the `eval_sets=` override so the config's `df21_eval` entry is used.

## 3. Video module (needs FaceForensics++ access)
Put the downloaded `c23` videos and the official `splits/` folder in
`MyDrive/forensight/datasets/FaceForensics++/`.

In [ ]:
# faces -> local disk (one-off; takes a while). Use num_frames=8 for a quick first pass.
!python -m forensight.data.video_preprocess --config configs/video_effnet.yaml data.num_frames=8

In [ ]:
# smoke run on a small backbone, then the real run
!python -m forensight.training.train_video --config configs/video_effnet.yaml train.epochs=1 model.backbone=efficientnet_b0 data.image_size=224

In [ ]:
!python -m forensight.training.train_video --config configs/video_effnet.yaml